# Fed-Batch Operating Mode Test Notebook

This notebook tests the **FEDBATCH** operating mode implementation.

**Test Coverage:**
- Simulation setup and execution
- Single batch run with initial conditions
- Multi-batch run with auto-extracted initial conditions
- Data integrity validation
- Performance metrics
- Fed-batch-specific validations (feed strategies, volume increase)

## 1. Setup and Imports

In [1]:
import sys
import numpy as np
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.gas_controllers import GasController
from animakinetica.models.stoichiometry import StoichiometryMode
from animakinetica.models.perfusion_strategies import PerfusionStrategy
print("✓ Imports successful")

✓ Imports successful


## 2. Load Parameters and Initialize Simulation

In [2]:
# Load base parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "excess_kinetics_adjusted", "fed_batch"])

# Initialize simulation with FEDBATCH mode
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.EXCESS,
    stoichiometry_mode=StoichiometryMode.DYNAMIC,  # Use constant stoichiometry for fed-batch
    operating_mode=OperatingMode.FEDBATCH,
    feed_strategy=FeedStrategy.CONSTANT,  
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False,  # Store detailed data for testing
    solver_method="BDF"  # Use BDF solver for stiff systems
)

print(f"✓ Simulation initialized")
print(f"  Operating Mode: {sim.operating_mode}")
print(f"  Kinetics Model: {sim.kinetics}")
print(f"  Feed Strategy: {sim.feed_strategy}")
print(f"  Gas Controller: {sim.gas_controller}")
print(f"  Stoichiometry Mode: {sim.stoichiometry_mode}")    

Loading parameters from 6 file(s) sequentially...
  [1/6] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\excess_kinetics\excess_kinetics_adjusted.yaml
  [6/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Tec

## 4. Multi-Batch Run (Auto-Extracted Initial Conditions)

In [3]:

# Run 5 repeated batches
num_batches = 1
multi_result = sim.run_simulation(number_of_batches=num_batches)

print(f"✓ {num_batches} batches completed")
print(f"\nSummary Statistics:")
print(f"  Average batch duration: {multi_result.projected_totals_stats['total_simulation_time_h']/num_batches:.2f} h")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.3f} kg/m3/day")

print(f"\nBatch-by-Batch Results:")
for i, batch in enumerate(sim.data.batches, 1):
    V_increase = batch.V_final - batch.initial_conditions[5]
    print(f"  Batch {i}: {batch.t_end:.2f} h, Xv={batch.Xv_final:.2e} cells/L, ΔV={V_increase:.2f} L")


run_simulation execution time: 122.162 ms
✓ 1 batches completed

Summary Statistics:
  Average batch duration: 119.04 h
  STY: 0.154 kg/m3/day

Batch-by-Batch Results:
  Batch 1: 119.04 h, Xv=9.63e+09 cells/L, ΔV=458.46 L


## 5. Data Integrity Validation

In [4]:
# Test batch 1 data integrity
batch_1 = sim.data.batches[0]

print("Data Integrity Checks for Batch 1:")
print("Termination reason:", batch_1.termination_reason)

print("\n1. State Variables:")
# Explicit index mapping matches y_points array: [Xv, C_Glc, C_Gln, C_Lac, C_Amm, V, C_O2, C_CT, C_H, C_Na, C_AA, C_Osm_bg]
state_var_indices = {
    'Xv': 0, 'C_Glc': 1, 'C_Gln': 2, 'C_Lac': 3, 'C_Amm': 4,
    'V': 5, 'C_O2': 6, 'C_CT': 7, 'C_H': 8, 'C_Na': 9, 'C_AA': 10,
    'C_Osm_bg': 11
}
for var, idx in state_var_indices.items():
    data = np.asarray(batch_1.y_points[idx], dtype=float)
    print(f"  {var} (idx {idx}): {len(data)} points, range [{data.min():.2e}, {data.max():.2e}]")
    assert len(data) > 0, f"{var} data is empty"
    assert not np.any(np.isnan(data)), f"{var} contains NaN values"

print("\n2. Kinetic Rates:")
kinetic_rates = ['mu', 'q_Glc', 'q_Gln', 'q_Lac', 'q_Amm', 'q_O2', 'q_CO2', 'q_AA']
for rate in kinetic_rates:
    rate_list = getattr(batch_1, f'{rate}_list')
    print(f"  {rate}: {len(rate_list)} points, range [{rate_list.min():.2e}, {rate_list.max():.2e}]")
    assert len(rate_list) > 0, f"{rate}_list is empty"
    assert not np.any(np.isnan(rate_list)), f"{rate}_list contains NaN values"

print("\n3. Nutrient Factors:")
nf_vars = ['NF_glucose', 'NF_glutamine', 'NF_O2', 'NF']
for nf in nf_vars:
    nf_list = getattr(batch_1, f'{nf}_list')
    print(f"  {nf}: {len(nf_list)} points, range [{nf_list.min():.2e}, {nf_list.max():.2e}]")
    assert len(nf_list) > 0, f"{nf}_list is empty"
    assert np.all((nf_list >= 0) & (nf_list <= 1)), f"{nf}_list contains values outside [0,1]"

print("\n4. Inhibition Factors:")
if_vars = ['IF_O2', 'IF_CO2', 'IF_lactate', 'IF_ammonia', 'IF_osmolality', 'IF']
for ifvar in if_vars:
    if_list = getattr(batch_1, f'{ifvar}_list')
    print(f"  {ifvar}: {len(if_list)} points, range [{if_list.min():.2e}, {if_list.max():.2e}]")
    assert len(if_list) > 0, f"{ifvar}_list is empty"
    assert np.all((if_list >= 0) & (if_list <= 1)), f"{ifvar}_list contains values outside [0,1]"

print("\n5. Gas Control Variables:")
gas_vars = ['N', 'Fg', 'y_O2', 'kLa', 'OTR', 'CTR', 'C_CO2']
for gv in gas_vars:
    gv_list = getattr(batch_1, f'{gv}_list')
    print(f"  {gv}: {len(gv_list)} points, range [{gv_list.min():.2e}, {gv_list.max():.2e}]")
    assert len(gv_list) > 0, f"{gv}_list is empty"
    assert not np.any(np.isnan(gv_list)), f"{gv}_list contains NaN values"

print("\n6. pH Control Variables:")
print(f"  D_base: {len(batch_1.D_base_list)} points, range [{batch_1.D_base_list.min():.2e}, {batch_1.D_base_list.max():.2e}]")
assert len(batch_1.D_base_list) > 0, "D_base_list is empty"

print("\n✓ All data integrity checks passed")


Data Integrity Checks for Batch 1:
Termination reason: Glucose depletion

1. State Variables:
  Xv (idx 0): 168 points, range [6.00e+08, 9.94e+09]
  C_Glc (idx 1): 168 points, range [1.00e-03, 2.50e+01]
  C_Gln (idx 2): 168 points, range [5.28e+00, 1.00e+01]
  C_Lac (idx 3): 168 points, range [0.00e+00, 8.94e+00]
  C_Amm (idx 4): 168 points, range [0.00e+00, 2.90e+00]
  V (idx 5): 168 points, range [6.00e+02, 1.06e+03]
  C_O2 (idx 6): 168 points, range [1.00e-02, 1.00e-02]
  C_CT (idx 7): 168 points, range [1.88e+01, 2.90e+01]
  C_H (idx 8): 168 points, range [6.31e-08, 6.31e-08]
  C_Na (idx 9): 168 points, range [0.00e+00, 0.00e+00]
  C_AA (idx 10): 168 points, range [2.56e-01, 5.00e+00]
  C_Osm_bg (idx 11): 168 points, range [2.50e+02, 2.50e+02]

2. Kinetic Rates:
  mu: 168 points, range [6.00e-05, 3.04e-02]
  q_Glc: 168 points, range [4.77e-11, 6.38e-11]
  q_Gln: 168 points, range [4.46e-12, 1.42e-11]
  q_Lac: 168 points, range [0.00e+00, 3.41e-11]
  q_Amm: 168 points, range [7.17e-

## 6. Performance Metrics

In [5]:
print("Performance Metrics (5-Batches):")
print(f"\n Total Biomass: {multi_result.projected_totals_stats['total_DCM_out_kg']:.2f} kg")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.3f} kg/m3/day")
print(f"  Medium Conversion Efficiency: {multi_result.summary_stats['medium_conversion_efficiency_m3_kg']:.3f} m3/kg DCM")
print(f"  Protein Conversion Efficiency: {multi_result.summary_stats['protein_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Feed Conversion Efficiency: {multi_result.summary_stats['feed_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Total Simulation Time: {multi_result.projected_totals_stats['total_simulation_time_h']:.2f} h")
print(f"  Number of Batches: {multi_result.projected_totals_stats['number_of_batches']}")


# Validate metrics are reasonable
assert multi_result.projected_totals_stats['total_DCM_out_kg'] > 0, "Total DCM should be positive"
assert multi_result.summary_stats['STY_kg_m3_day'] > 0, "STY should be positive"
assert 0 < multi_result.summary_stats['medium_conversion_efficiency_m3_kg'] < 1000, "Medium conversion efficiency should be positive and reasonable"
assert multi_result.summary_stats['protein_conversion_efficiency_kg_kg'] > 0, "Protein conversion efficiency should be positive"
assert multi_result.summary_stats['feed_conversion_efficiency_kg_kg'] > 0, "Feed conversion efficiency should be positive"
assert multi_result.projected_totals_stats['total_simulation_time_h'] > 0, "Total simulation time should be positive"
assert multi_result.projected_totals_stats['number_of_batches'] == num_batches, f"Number of batches should be {num_batches}"

print("\n✓ Performance metrics validated")


Performance Metrics (5-Batches):

 Total Biomass: 1.53 kg
  STY: 0.154 kg/m3/day
  Medium Conversion Efficiency: 0.692 m3/kg DCM
  Protein Conversion Efficiency: 2.525 kg/kg
  Feed Conversion Efficiency: 4.417 kg/kg
  Total Simulation Time: 119.04 h
  Number of Batches: 1

✓ Performance metrics validated


## 7. Fed-Batch-Specific Validations

In [6]:
print("Fed-Batch-Specific Validations:")

# 1. Volume should increase over time (feeding occurs)
print("\n1. Volume Increase:")
for i, batch in enumerate(sim.data.batches, 1):
    V_initial = float(np.asarray(batch.y_points[5, 0], dtype=float))  # State 5 (V), first timepoint
    V_final = float(np.asarray(batch.y_points[5, -1], dtype=float))   # State 5 (V), last timepoint
    V_increase = V_final - V_initial
    V_increase_pct = (V_increase / V_initial) * 100
    print(f"  Batch {i}: V_initial={V_initial:.2f} L, V_final={V_final:.2f} L, ΔV={V_increase:.2f} L ({V_increase_pct:.1f}%)")
    assert V_increase > 0, f"Batch {i}: Volume did not increase (no feeding detected)"

print("  ✓ Volume increases in all batches (feeding confirmed)")

# 2. Feed rate calculation and consistency
print("\n2. Feed Rate Profile (Batch 1):")
t_points = batch_1.t_points  # Time array from solve_ivp
V_points = np.asarray(batch_1.y_points[5, :], dtype=float)  # State 5 (V), all timepoints
# Calculate instantaneous feed rate from dV/dt
feed_rate_calc = np.diff(V_points) / np.diff(t_points)
print(f"  Average feed rate: {feed_rate_calc.mean():.4f} L/h")
print(f"  Feed rate range: [{feed_rate_calc.min():.4f}, {feed_rate_calc.max():.4f}] L/h")
if sim.feed_strategy == FeedStrategy.EXPONENTIAL:
    print(f"  Feed strategy: Exponential (should increase over time)")
    # Check if feed rate increases (at least in first half)
    mid_point = len(feed_rate_calc) // 2
    early_avg = feed_rate_calc[:mid_point].mean()
    late_avg = feed_rate_calc[mid_point:].mean()
    print(f"  Early avg: {early_avg:.4f} L/h, Late avg: {late_avg:.4f} L/h")
    assert late_avg > early_avg * 0.9, "Exponential feed should increase or remain similar"

# 3. Substrate maintenance above depletion threshold
print("\n3. Substrate Levels:")
for i, batch in enumerate(sim.data.batches, 1):
    C_Glc_min = float(np.asarray(batch.y_points[1, :], dtype=float).min())  # State 1 (C_Glc)
    C_Gln_min = float(np.asarray(batch.y_points[2, :], dtype=float).min())  # State 2 (C_Gln)
    C_Glc_final = float(np.asarray(batch.y_points[1, -1], dtype=float))    # State 1, last timepoint
    C_Gln_final = float(np.asarray(batch.y_points[2, -1], dtype=float))    # State 2, last timepoint
    print(f"  Batch {i}:")
    print(f"    Glucose: min={C_Glc_min:.2f} mmol/L, final={C_Glc_final:.2f} mmol/L")
    print(f"    Glutamine: min={C_Gln_min:.2f} mmol/L, final={C_Gln_final:.2f} mmol/L")
    print(f"    Termination: {batch.termination_reason}")

# 4. Cell growth dynamics
print("\n4. Cell Growth Dynamics:")
for i, batch in enumerate(sim.data.batches, 1):
    Xv_initial = float(np.asarray(batch.y_points[0, 0], dtype=float))   # State 0 (Xv), first timepoint
    Xv_final = float(np.asarray(batch.y_points[0, -1], dtype=float))    # State 0 (Xv), last timepoint
    Xv_max = float(np.asarray(batch.y_points[0, :], dtype=float).max()) # State 0 (Xv), all timepoints
    fold_increase = Xv_max / Xv_initial
    print(f"  Batch {i}: Xv_max={Xv_max:.2e} cells/L, Fold increase={fold_increase:.2f}x")
    assert Xv_max > Xv_initial, f"Batch {i}: No cell growth detected"

print("  ✓ Cell growth observed in all batches")

# 5. Maximum volume event check
print("\n5. Maximum Volume Check:")
V_reactor = float(params.reactor.V_reactor)
for i, batch in enumerate(sim.data.batches, 1):
    V_max_reached = float(np.asarray(batch.y_points[5, :], dtype=float).max())  # State 5 (V)
    print(f"  Batch {i}: V_max={V_max_reached:.2f} L (reactor capacity={V_reactor:.2f} L)")
    print(f"    Termination: {batch.termination_reason}")
    assert V_max_reached <= V_reactor, f"Batch {i}: Volume exceeded reactor capacity"

print("\n✓ All fed-batch-specific validations passed")

Fed-Batch-Specific Validations:

1. Volume Increase:
  Batch 1: V_initial=600.00 L, V_final=1058.46 L, ΔV=458.46 L (76.4%)
  ✓ Volume increases in all batches (feeding confirmed)

2. Feed Rate Profile (Batch 1):
  Average feed rate: 5.6048 L/h
  Feed rate range: [0.0000, 18.0000] L/h

3. Substrate Levels:
  Batch 1:
    Glucose: min=0.00 mmol/L, final=0.00 mmol/L
    Glutamine: min=5.28 mmol/L, final=5.31 mmol/L
    Termination: Glucose depletion

4. Cell Growth Dynamics:
  Batch 1: Xv_max=9.94e+09 cells/L, Fold increase=16.56x
  ✓ Cell growth observed in all batches

5. Maximum Volume Check:
  Batch 1: V_max=1058.46 L (reactor capacity=2500.00 L)
    Termination: Glucose depletion

✓ All fed-batch-specific validations passed


In [7]:
sim.data.plot_biomass_metabolites()

In [8]:
sim.data.plot_volume_feed()

In [9]:
sim.data.plot_pH_control(batch_number=1)

In [10]:
sim.data.plot_gas_transfer()

In [11]:
sim.data.plot_kinetics()

## 8. Test Different Feed Strategies

In [12]:
print("Testing Different Feed Strategies:\n")

import copy
from animakinetica.core.parameters import Parameter

feed_strategies = [
    FeedStrategy.CONSTANT,
    FeedStrategy.EXPONENTIAL,
    FeedStrategy.LINEAR,
    FeedStrategy.CSFR,
    FeedStrategy.GLUCOSE_SETPOINT,
]
results = {}

for feed_strat in feed_strategies:
    print(f"\nTesting {feed_strat}...")
    params_test = copy.deepcopy(params)
    params_test.update("excess_kinetics_batch")
    sim_test = CellCultureSimulation(
        parameters=params_test,
        kinetics=Kinetics.EXCESS,
        operating_mode=OperatingMode.FEDBATCH,
        feed_strategy=feed_strat,
        gas_controller=GasController.CASCADE_N_FG_XO2,
        performance_mode=True,  # Fast mode for comparison
        solver_method="BDF"  # Use BDF solver for stiff systems
    )
    result = sim_test.run_simulation(number_of_batches=1)

    # Extract batch data from SimulationData object
    batch = result.batches[0]
    t_points = np.asarray(batch.t_points, dtype=float)
    V_points = np.asarray(batch.y_points[5, :], dtype=float)
    feed_profile = np.diff(V_points) / np.diff(t_points)
    max_feed = float(np.max(feed_profile)) if len(feed_profile) > 0 else np.nan
    mean_feed = float(np.mean(feed_profile)) if len(feed_profile) > 0 else np.nan

    results[str(feed_strat)] = {
        't_end': batch.t_end,
        'Xv_final': batch.Xv_final,
        'V_final': batch.V_final,
        'termination': batch.termination_reason,
        'feed_mean': mean_feed,
        'feed_max': max_feed,
    }

    print(f"  Duration: {batch.t_end:.2f} h, Xv_final: {batch.Xv_final:.2e} cells/L, V_final: {batch.V_final:.2f} L")
    print(f"  Feed mean/max: {mean_feed:.4f}/{max_feed:.4f} L/h")
    print(f"  Termination: {batch.termination_reason}")

print("\n✓ All feed strategies tested successfully")


Testing Different Feed Strategies:


Testing Constant...
run_simulation execution time: 165.581 ms
  Duration: 123.65 h, Xv_final: 7.07e+09 cells/L, V_final: 1104.21 L
  Feed mean/max: 7.0728/18.4483 L/h
  Termination: Ammonia ic100

Testing Exponential...
run_simulation execution time: 142.786 ms
  Duration: 142.77 h, Xv_final: 5.55e+09 cells/L, V_final: 2000.00 L
  Feed mean/max: 9.5852/45.7024 L/h
  Termination: Max volume

Testing Linear...
run_simulation execution time: 111.103 ms
  Duration: 128.19 h, Xv_final: 6.64e+09 cells/L, V_final: 1237.86 L
  Feed mean/max: 7.0418/21.3450 L/h
  Termination: Ammonia ic100

Testing Csfr...
run_simulation execution time: 93.937 ms
  Duration: 135.21 h, Xv_final: 5.96e+09 cells/L, V_final: 1459.43 L
  Feed mean/max: 7.6121/26.1432 L/h
  Termination: Ammonia ic100

Testing Glucose Setpoint...
run_simulation execution time: 99.259 ms
  Duration: 141.47 h, Xv_final: 5.77e+09 cells/L, V_final: 2000.00 L
  Feed mean/max: 10.5048/43.0075 L/h
  Termi

In [13]:
# Comparison plot: feed profiles and glucose concentration across all feed strategies
import copy
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from animakinetica.analysis.batch_visualization import (
    plot_batch_volume_feed,
    plot_batch_biomass_metabolites,
 )

# Use same simulation setup for fair comparison
feed_strategies_compare = [
    FeedStrategy.CONSTANT,
    FeedStrategy.EXPONENTIAL,
    FeedStrategy.LINEAR,
    FeedStrategy.CSFR,
    FeedStrategy.GLUCOSE_SETPOINT,
]

# Optional color mapping for consistent styling
color_map = {
    FeedStrategy.CONSTANT: "#1f77b4",
    FeedStrategy.EXPONENTIAL: "#ff7f0e",
    FeedStrategy.LINEAR: "#2ca02c",
    FeedStrategy.CSFR: "#d62728",
    FeedStrategy.GLUCOSE_SETPOINT: "#9467bd",
}

# Build one comparison figure
fig_compare = make_subplots(
    rows=1, cols=5,
    subplot_titles=(
        "Process Feed Rate Comparison",
        "Base Feed Rate Comparison",
        "Total Inlet Rate Comparison",
        "Volume Comparison",
        "Glucose Comparison",
    ),
    horizontal_spacing=0.08
)

for fs in feed_strategies_compare:
    params_test = copy.deepcopy(params)
    params_test.update("excess_kinetics_batch")  # Use excess kinetics for more dynamic feed profiles
    #params_test.update("excess_kinetics_batch")
    sim_test = CellCultureSimulation(
        parameters=params_test,
        kinetics=Kinetics.EXCESS,
        operating_mode=OperatingMode.FEDBATCH,
        feed_strategy=fs,
        gas_controller=GasController.CASCADE_N_FG_XO2,
        performance_mode=False,  # needed so detailed lists are available for plotting
        solver_method="BDF"
    )

    res = sim_test.run_simulation(number_of_batches=1)

    # Reuse existing plotting utilities and extract traces from their outputs
    fig_volume = plot_batch_volume_feed(res, batch_number=1, layout_kwargs={"showlegend": False})
    fig_biomass = plot_batch_biomass_metabolites(res, batch_number=1, layout_kwargs={"showlegend": False})

    total_inlet_trace = next(tr for tr in fig_volume.data if tr.name == "Total Inlet Rate")
    volume_trace = next(tr for tr in fig_volume.data if tr.name == "Volume")
    base_inlet_trace = next(tr for tr in fig_volume.data if tr.name == "Base Feed Rate")
    feed_inlet_trace = next(tr for tr in fig_volume.data if tr.name == "Process Feed Rate")
    glucose_trace = next(tr for tr in fig_biomass.data if tr.name == "Glucose")

    label = fs.name
    color = color_map.get(fs, None)

    fig_compare.add_trace(
        go.Scatter(
            x=feed_inlet_trace.x,
            y=feed_inlet_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=True,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_feed: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=1
    )

    fig_compare.add_trace(
        go.Scatter(
            x=base_inlet_trace.x,
            y=base_inlet_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_base: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=2
    )

    fig_compare.add_trace(
        go.Scatter(
            x=total_inlet_trace.x,
            y=total_inlet_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_total: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=3
    )

    fig_compare.add_trace(
        go.Scatter(
            x=volume_trace.x,
            y=volume_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color, dash="dot"),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>V: %{y:.2f} L<extra></extra>",
        ),
        row=1, col=4
    )

    fig_compare.add_trace(
        go.Scatter(
            x=glucose_trace.x,
            y=glucose_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>Glucose: %{y:.2f} mmol/L<extra></extra>",
        ),
        row=1, col=5
    )

for col in range(1, 6):
    fig_compare.update_xaxes(title_text="Time (h)", row=1, col=col)

fig_compare.update_yaxes(title_text="Process Feed Rate (L/h)", row=1, col=1)
fig_compare.update_yaxes(title_text="Base Feed Rate (L/h)", row=1, col=2)
fig_compare.update_yaxes(title_text="Total Inlet Rate (L/h)", row=1, col=3)
fig_compare.update_yaxes(title_text="Volume (L)", row=1, col=4)
fig_compare.update_yaxes(title_text="Glucose (mmol/L)", row=1, col=5)

fig_compare.update_layout(
    title_text="Fed-Batch Feed Strategy Comparison (Same Simulation Setup)",
    template="plotly_white",
    height=500,
    width=3200,
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
)

fig_compare.show()

run_simulation execution time: 168.964 ms
run_simulation execution time: 90.143 ms
run_simulation execution time: 121.378 ms
run_simulation execution time: 331.532 ms
run_simulation execution time: 103.027 ms


In [14]:
data = res.batches

fig_pH_plots = res.plot_pH_control(batch_number=1)
fig_pH_plots.show()

In [15]:

fig_biomass_metabolites = plot_batch_biomass_metabolites(res, batch_number=1)
fig_biomass_metabolites.show()

In [16]:
fig_kinetics = res.plot_kinetics()
fig_kinetics.show()